# Training with Non-Wrapping Mode

This tutorial demonstrates how to use Opacus's non-wrapping mode (`wrap_model=False`), which provides better compatibility with transformer models and other complex architectures by avoiding model wrapping.

## What is Non-Wrapping Mode?

By default, Opacus wraps your model in a `GradSampleModule` to compute per-sample gradients. This wrapper can cause issues:
- **Type checking**: `isinstance(model, MyModel)` fails after wrapping
- **State dict**: Keys get `_module.` prefix, complicating checkpoint loading
- **Attribute access**: Models with custom `__getattr__` (e.g., HuggingFace Transformers) may break

Non-wrapping mode attaches hooks **directly to your model** without wrapping it, keeping the model intact.

## Setup

First, let's import libraries and create a simple dataset:

In [ ]:
import warnings
warnings.simplefilter("ignore")

import torch
from torch import nn, optim
from torch.utils.data import TensorDataset, DataLoader

# Create synthetic dataset
n_samples = 1000
n_features = 20
n_classes = 10

X = torch.randn(n_samples, n_features)
y = torch.randint(0, n_classes, (n_samples,))

dataset = TensorDataset(X, y)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

## Define a Model

Let's create a simple classifier:

In [ ]:
class SimpleClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, output_dim)
        self.relu = nn.ReLU()
    
    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.fc3(x)
        return x

model = SimpleClassifier(n_features, 64, n_classes)
print(f"Model type: {type(model).__name__}")
print(f"isinstance check: {isinstance(model, SimpleClassifier)}")

## Comparison: Wrapped vs Non-Wrapped

Let's compare the default wrapped mode with non-wrapping mode:

In [ ]:
from opacus import PrivacyEngine

# === Default wrapped mode ===
model_wrapped = SimpleClassifier(n_features, 64, n_classes)
optimizer_wrapped = optim.Adam(model_wrapped.parameters(), lr=0.001)

privacy_engine = PrivacyEngine()
model_wrapped, optimizer_wrapped, dataloader_wrapped = privacy_engine.make_private(
    module=model_wrapped,
    optimizer=optimizer_wrapped,
    data_loader=dataloader,
    noise_multiplier=1.0,
    max_grad_norm=1.0,
    # wrap_model=True is the default
)

print("=== Wrapped Mode (default) ===")
print(f"Model type: {type(model_wrapped).__name__}")
print(f"isinstance check: {isinstance(model_wrapped, SimpleClassifier)}")
print(f"State dict keys (first 3): {list(model_wrapped.state_dict().keys())[:3]}")
print()

In [ ]:
# === Non-wrapping mode ===
model_nowrap = SimpleClassifier(n_features, 64, n_classes)
optimizer_nowrap = optim.Adam(model_nowrap.parameters(), lr=0.001)

privacy_engine2 = PrivacyEngine()
hooks, optimizer_nowrap, dataloader_nowrap = privacy_engine2.make_private(
    module=model_nowrap,
    optimizer=optimizer_nowrap,
    data_loader=dataloader,
    noise_multiplier=1.0,
    max_grad_norm=1.0,
    wrap_model=False,  # Enable non-wrapping mode
)

print("=== Non-Wrapping Mode ===")
print(f"Hooks type: {type(hooks).__name__}")
print(f"Model type: {type(model_nowrap).__name__}")
print(f"isinstance check: {isinstance(model_nowrap, SimpleClassifier)}")
print(f"State dict keys (first 3): {list(model_nowrap.state_dict().keys())[:3]}")
print(f"Model unchanged - use it directly!")

**Key differences:**
- Wrapped: Model becomes `GradSampleModule`, `isinstance()` fails, keys have `_module.` prefix
- Non-wrapped: Model stays `SimpleClassifier`, `isinstance()` works, clean state dict keys

## Important: Your Model Is Unchanged

In non-wrapping mode, `make_private` returns a **hooks object** for cleanup. Your model is unchanged - just use it!

```python
# Your model
model = SimpleClassifier(...)

# Make private returns hooks
hooks, optimizer, dataloader = privacy_engine.make_private(
    module=model,
    wrap_model=False,
    ...
)

# ✅ CORRECT: Use your model directly
output = model(input)              # You already have the model!
state_dict = model.state_dict()   # Use it normally
model.train()                      # Nothing changed

# ❌ WRONG: Don't use hooks for model operations
# hooks.state_dict()               # This will fail!
# hooks(input)                     # This will fail!
```

The `hooks` object is **only** for cleanup: `hooks.cleanup()`

Your model is untouched and works exactly as before.

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_nowrap = model_nowrap.to(device)
criterion = nn.CrossEntropyLoss()

EPOCHS = 3
DELTA = 1e-5

for epoch in range(EPOCHS):
    model_nowrap.train()
    total_loss = 0
    
    for batch_idx, (data, target) in enumerate(dataloader_nowrap):
        data, target = data.to(device), target.to(device)
        
        optimizer_nowrap.zero_grad()
        output = model_nowrap(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer_nowrap.step()
        
        total_loss += loss.item()
    
    epsilon = privacy_engine2.get_epsilon(DELTA)
    avg_loss = total_loss / len(dataloader_nowrap)
    print(f"Epoch {epoch + 1}/{EPOCHS} | Loss: {avg_loss:.4f} | ε: {epsilon:.2f} (δ={DELTA})")

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_nowrap = model_nowrap.to(device)
criterion = nn.CrossEntropyLoss()

EPOCHS = 3
DELTA = 1e-5

for epoch in range(EPOCHS):
    model_nowrap.train()
    total_loss = 0
    
    for batch_idx, (data, target) in enumerate(dataloader_nowrap):
        data, target = data.to(device), target.to(device)
        
        optimizer_nowrap.zero_grad()
        output = model_nowrap(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer_nowrap.step()
        
        total_loss += loss.item()
    
    epsilon = privacy_engine2.get_epsilon(DELTA)
    avg_loss = total_loss / len(dataloader_nowrap)
    print(f"Epoch {epoch + 1}/{EPOCHS} | Loss: {avg_loss:.4f} | ε: {epsilon:.2f} (δ={DELTA})")

# Clean up hooks
hooks.cleanup()
print("Hooks cleaned up successfully")

# Model is now back to its original state
print(f"Model can now be used normally")

In [ ]:
# Clean up hooks
if hasattr(model_nowrap, '_opacus_hooks'):
    model_nowrap._opacus_hooks.cleanup()
    delattr(model_nowrap, '_opacus_hooks')
    print("Hooks cleaned up successfully")

# Now the model is back to its original state
print(f"Has _opacus_hooks: {hasattr(model_nowrap, '_opacus_hooks')}")
print(f"Model can now be used normally")

## What Does Cleanup Do?

The `cleanup()` method:
1. **Removes all hooks** attached during `make_private()`
2. **Deletes monkeypatched attributes** from parameters (e.g., `grad_sample`, `_forward_counter`)
3. **Restores model to original state** as if Opacus was never used

Without cleanup, these hooks and attributes remain, which can:
- Cause memory leaks
- Interfere with subsequent training
- Lead to unexpected errors

# Save checkpoint (train a fresh model first)
model_save = SimpleClassifier(n_features, 64, n_classes)
optimizer_save = optim.Adam(model_save.parameters(), lr=0.001)

privacy_engine3 = PrivacyEngine()
hooks_save, optimizer_save, dataloader_save = privacy_engine3.make_private(
    module=model_save,
    optimizer=optimizer_save,
    data_loader=dataloader,
    noise_multiplier=1.0,
    max_grad_norm=1.0,
    wrap_model=False,
)

# Use model_save directly - it's unchanged!
# Save state dict - clean keys without _module. prefix
torch.save({
    'model_state_dict': model_save.state_dict(),
    'optimizer_state_dict': optimizer_save.state_dict(),
}, 'checkpoint.pt')
print("Checkpoint saved with clean state dict keys")

# Clean up
hooks_save.cleanup()

In [ ]:
# Load checkpoint into a new model
model_load = SimpleClassifier(n_features, 64, n_classes)
optimizer_load = optim.Adam(model_load.parameters(), lr=0.001)

checkpoint = torch.load('checkpoint.pt')
model_load.load_state_dict(checkpoint['model_state_dict'])  # No _module. prefix issues!
optimizer_load.load_state_dict(checkpoint['optimizer_state_dict'])
print("Checkpoint loaded successfully")

# Continue training with DP if needed
privacy_engine4 = PrivacyEngine()
hooks_load, optimizer_load, dataloader_load = privacy_engine4.make_private(
    module=model_load,
    optimizer=optimizer_load,
    data_loader=dataloader,
    noise_multiplier=1.0,
    max_grad_norm=1.0,
    wrap_model=False,
)
print("Ready to continue training")

In [ ]:
# Load checkpoint into a new model
model_load = SimpleClassifier(n_features, 64, n_classes)
optimizer_load = optim.Adam(model_load.parameters(), lr=0.001)

checkpoint = torch.load('checkpoint.pt')
model_load.load_state_dict(checkpoint['model_state_dict'])  # No _module. prefix issues!
optimizer_load.load_state_dict(checkpoint['optimizer_state_dict'])
print("Checkpoint loaded successfully")

# Continue training with DP if needed
privacy_engine4 = PrivacyEngine()
model_load, optimizer_load, dataloader_load = privacy_engine4.make_private(
    module=model_load,
    optimizer=optimizer_load,
    data_loader=dataloader,
    noise_multiplier=1.0,
    max_grad_norm=1.0,
    wrap_model=False,
)
print("Ready to continue training")

model_eps = SimpleClassifier(n_features, 64, n_classes)
optimizer_eps = optim.Adam(model_eps.parameters(), lr=0.001)
dataloader_eps = DataLoader(dataset, batch_size=32, shuffle=True)

privacy_engine5 = PrivacyEngine()
hooks_eps, optimizer_eps, dataloader_eps = privacy_engine5.make_private_with_epsilon(
    module=model_eps,
    optimizer=optimizer_eps,
    data_loader=dataloader_eps,
    target_epsilon=3.0,
    target_delta=1e-5,
    epochs=EPOCHS,
    max_grad_norm=1.0,
    wrap_model=False,  # Works with non-wrapping mode too!
)

print(f"Target epsilon: 3.0")
print(f"Computed noise multiplier: {optimizer_eps.noise_multiplier:.3f}")

# Don't forget cleanup later!
# hooks_eps.cleanup()

In [ ]:
model_eps = SimpleClassifier(n_features, 64, n_classes)
optimizer_eps = optim.Adam(model_eps.parameters(), lr=0.001)
dataloader_eps = DataLoader(dataset, batch_size=32, shuffle=True)

privacy_engine5 = PrivacyEngine()
model_eps, optimizer_eps, dataloader_eps = privacy_engine5.make_private_with_epsilon(
    module=model_eps,
    optimizer=optimizer_eps,
    data_loader=dataloader_eps,
    target_epsilon=3.0,
    target_delta=1e-5,
    epochs=EPOCHS,
    max_grad_norm=1.0,
    wrap_model=False,  # Works with non-wrapping mode too!
)

print(f"Target epsilon: 3.0")
print(f"Computed noise multiplier: {optimizer_eps.noise_multiplier:.3f}")

# Don't forget cleanup later!
# model_eps._opacus_hooks.cleanup()

## When to Use Non-Wrapping Mode?

**Use `wrap_model=False` when:**
- Working with **HuggingFace Transformers** or models with custom `__getattr__`
- You need **`isinstance()` checks** to work correctly
- You want **clean state dicts** without `_module.` prefixes
- Your pipeline relies on **model type introspection**

**Use default `wrap_model=True` when:**
- You have simple models without complex introspection needs
- You want **automatic cleanup** (wrapper discarded when out of scope)
- You don't need the benefits above

**Note:** ExpandedWeights mode (`grad_sample_mode="ew"`) is **not supported** with non-wrapping mode, as it requires overriding the model's `.forward()` method.

## Summary

| Feature | Wrapped Mode | Non-Wrapping Mode |
|---------|--------------|------------------|
| API parameter | `wrap_model=True` (default) | `wrap_model=False` |
| Model type preserved | ❌ No | ✅ Yes |
| `isinstance()` works | ❌ No | ✅ Yes |
| State dict keys | `_module.` prefix | ✅ Clean |
| Cleanup required | ❌ Automatic | ⚠️ Manual (`.cleanup()`) |
| HuggingFace compatibility | 🟨 May have issues | ✅ Better |
| ExpandedWeights support | ✅ Yes | ❌ No |

**Key takeaway:** Non-wrapping mode is a practical option for models where wrapping causes compatibility issues. Just remember to clean up!